# Chapter 9 §9.2: Invoice Entity Extraction

This notebook accompanies Chapter 9 §9.2 of *Context Engineering with DSPy*: **Invoice Entity Extraction**.

**Required environment variables (set in your `.env`):**

- `OPENAI_API_KEY`

## Overview

```
9.2 Invoice Entity Extraction

Demonstrates a progressive optimization pipeline:
1. Baseline — ChainOfThought extraction
2. + Few-shot examples — grounds the model on expected output format
3. + Majority voting — consensus across 3 parallel extractions
4. + GEPA instruction learning — optimizer discovers better extraction strategies

Each step shows accuracy AND cost, so you can see the tradeoff:
Majority voting triples your API spend. When is 3x cost worth it?

Failure mode: Context Distraction (Ch 1.4.3) + Token Costs (Ch 1.4.2)
Technique: FewShot Learning + Prompt Optimization (GEPA)
Agentic pattern: Parallelization (majority voting)

Usage:
    python invoice_extraction.py                  # full pipeline
    python invoice_extraction.py --baseline-only  # skip optimization
    python invoice_extraction.py --mode medium    # heavier GEPA optimization
```


In [ ]:
%pip install -r ../requirements.txt -q

## Imports and LM configuration

In [ ]:
import os
import json
from dotenv import load_dotenv

import dspy
from dspy.evaluate import Evaluate

load_dotenv()
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

## Signature — 8 standard invoice fields

In [ ]:
class InvoiceExtraction(dspy.Signature):
    """Extract structured fields from free-form invoice text.

    Return empty string for fields not present in the text.
    Do not hallucinate values — if a field is missing, leave it blank.
    """

    text: str = dspy.InputField(desc="Raw invoice text")
    rationale: str = dspy.OutputField(desc="Brief reasoning about which fields were detected")
    company: str = dspy.OutputField(desc="Company name (issuer/seller)")
    billed_to: str = dspy.OutputField(desc="Customer/recipient name")
    invoice_number: str = dspy.OutputField(desc="Invoice number or ID")
    invoice_date: str = dspy.OutputField(desc="Invoice issue date")
    total_amount: str = dspy.OutputField(desc="Total amount with currency symbol")
    bank_name: str = dspy.OutputField(desc="Bank name for payment")
    account_name: str = dspy.OutputField(desc="Account holder name")
    account_number: str = dspy.OutputField(desc="Account number or IBAN")

FIELD_KEYS = [
    "company", "billed_to", "invoice_number", "invoice_date",
    "total_amount", "bank_name", "account_name", "account_number",
]

## Dataset — diverse invoice formats

In [ ]:
INVOICES = [
    # Standard US invoice
    {
        "text": """INVOICE
Acme Corporation
Bill To: Globex Industries
Invoice No: INV-2024-001
Invoice Date: January 15, 2024
TOTAL: $1,250.00
Bank: First National Bank
Account Name: Acme Corporation
Account No: 1234-5678-9012""",
        "company": "Acme Corporation", "billed_to": "Globex Industries",
        "invoice_number": "INV-2024-001", "invoice_date": "January 15, 2024",
        "total_amount": "$1,250.00", "bank_name": "First National Bank",
        "account_name": "Acme Corporation", "account_number": "1234-5678-9012",
    },
    # European format with IBAN
    {
        "text": """SALES INVOICE
TechSupply Ltd
Customer: Downtown Retail
Inv#: TS-789
Date: 2024-02-20
Amount Due: €450.00
Payment to: Metro Bank
Beneficiary: TechSupply Ltd
IBAN: GB29 NWBK 6016 1331 9268 19""",
        "company": "TechSupply Ltd", "billed_to": "Downtown Retail",
        "invoice_number": "TS-789", "invoice_date": "2024-02-20",
        "total_amount": "€450.00", "bank_name": "Metro Bank",
        "account_name": "TechSupply Ltd", "account_number": "GB29 NWBK 6016 1331 9268 19",
    },
    # Messy format — labels in different positions
    {
        "text": """Creative Studios Inc.
INVOICE #CS-2024-042
Issue Date: March 5, 2024

To: Blue Sky Marketing

Services rendered: Brand identity package
TOTAL PAYABLE: $3,800.00

Bank Details:
Pacific Trust Bank
Account: Creative Studios Inc.
Acct#: 9876543210""",
        "company": "Creative Studios Inc.", "billed_to": "Blue Sky Marketing",
        "invoice_number": "CS-2024-042", "invoice_date": "March 5, 2024",
        "total_amount": "$3,800.00", "bank_name": "Pacific Trust Bank",
        "account_name": "Creative Studios Inc.", "account_number": "9876543210",
    },
    # Minimal invoice — some fields missing
    {
        "text": """TAX INVOICE
Building Supplies Co
Customer: HomeBuilder LLC
Invoice #: BSC-2024-123
Invoice Date: May 15, 2024
TOTAL: $12,450.00""",
        "company": "Building Supplies Co", "billed_to": "HomeBuilder LLC",
        "invoice_number": "BSC-2024-123", "invoice_date": "May 15, 2024",
        "total_amount": "$12,450.00", "bank_name": "", "account_name": "", "account_number": "",
    },
    # Multi-currency with non-standard labels
    {
        "text": """Green Energy Solutions
Billed To: City Council
Inv No: GES-555
Date: 2024-04-10
Grand Total: $5,200.00
Bank: Community Bank
Account Holder: Green Energy Solutions
Account: 5551234567""",
        "company": "Green Energy Solutions", "billed_to": "City Council",
        "invoice_number": "GES-555", "invoice_date": "2024-04-10",
        "total_amount": "$5,200.00", "bank_name": "Community Bank",
        "account_name": "Green Energy Solutions", "account_number": "5551234567",
    },
]

def build_dataset():
    """Convert to DSPy Examples with train/test split."""
    examples = [
        dspy.Example(text=item["text"], **{k: item[k] for k in FIELD_KEYS}).with_inputs("text")
        for item in INVOICES
    ]
    return examples[:3], examples[3:]

## Metric — partial credit for multi-field extraction

In [ ]:
def field_accuracy(example, pred, trace=None, pred_name=None, pred_trace=None):
    """Partial credit: +0.5 for field presence, +0.5 for correct value.

    This is more informative than binary pass/fail for extraction tasks.
    A model that gets 6/8 fields right scores higher than one that gets 2/8,
    even though both "failed" by a strict exact-match standard.

    The extra pred_name and pred_trace kwargs satisfy the GEPA feedback
    metric protocol, which passes per-predictor context during optimization.
    """
    present = 0
    correct = 0

    for field in FIELD_KEYS:
        gold = (getattr(example, field, "") or "").strip().lower()
        got = (getattr(pred, field, "") or "").strip().lower()

        if got:
            present += 1
        if gold == got:
            correct += 1

    return (0.5 * present + 0.5 * correct) / len(FIELD_KEYS)

## Pipeline stages

In [ ]:
def run_baseline(trainset, testset):
    """Stage 1: Raw ChainOfThought extraction."""
    print("=" * 60)
    print("STAGE 1: Baseline (ChainOfThought)")
    print("=" * 60)

    extractor = dspy.ChainOfThought(InvoiceExtraction)
    evaluator = Evaluate(devset=testset, metric=field_accuracy, display_progress=False)
    score = float(evaluator(extractor))

    print(f"  Accuracy: {score:.1f}%")
    print(f"  Cost: ~1 LLM call per invoice")
    return extractor, score

def run_fewshot(trainset, testset):
    """Stage 2: Add few-shot examples to ground the output format."""
    print("\n" + "=" * 60)
    print("STAGE 2: + Few-Shot Examples (BootstrapFewShot)")
    print("=" * 60)

    from dspy.teleprompt import BootstrapFewShot

    optimizer = BootstrapFewShot(
        metric=field_accuracy,
        max_bootstrapped_demos=2,
        max_labeled_demos=3,
    )

    extractor = optimizer.compile(
        dspy.ChainOfThought(InvoiceExtraction),
        trainset=trainset,
    )

    evaluator = Evaluate(devset=testset, metric=field_accuracy, display_progress=False)
    score = float(evaluator(extractor))

    print(f"  Accuracy: {score:.1f}%")
    print(f"  Cost: ~1 LLM call per invoice (few-shot examples add tokens, not calls)")
    return extractor, score

class MajorityVotingExtractor(dspy.Module):
    """Run extraction 3 times and take the most common value per field.

    dspy.majority is a post-hoc aggregation function, so we wrap it
    in a Module that calls the base extractor multiple times and
    picks the consensus answer for each field.
    """

    def __init__(self, base_extractor, num_preds=3):
        super().__init__()
        self.base = base_extractor
        self.num_preds = num_preds

    def forward(self, **kwargs):
        from collections import Counter

        preds = [self.base(**kwargs) for _ in range(self.num_preds)]

        # For each output field, pick the most common value
        consensus = {}
        for field in FIELD_KEYS:
            values = [(getattr(p, field, "") or "").strip() for p in preds]
            most_common = Counter(values).most_common(1)[0][0]
            consensus[field] = most_common

        return dspy.Prediction(**consensus)

def run_majority(trainset, testset):
    """Stage 3: Majority voting — run 3 extractions and take consensus."""
    print("\n" + "=" * 60)
    print("STAGE 3: + Majority Voting (3 parallel calls)")
    print("=" * 60)

    # Use the few-shot extractor as base, wrapped in majority voting
    from dspy.teleprompt import BootstrapFewShot

    optimizer = BootstrapFewShot(
        metric=field_accuracy,
        max_bootstrapped_demos=2,
        max_labeled_demos=3,
    )

    base = optimizer.compile(
        dspy.ChainOfThought(InvoiceExtraction),
        trainset=trainset,
    )

    # Run 3 extractions per invoice and take consensus per field
    majority_extractor = MajorityVotingExtractor(base, num_preds=3)

    evaluator = Evaluate(devset=testset, metric=field_accuracy, display_progress=False)
    score = float(evaluator(majority_extractor))

    print(f"  Accuracy: {score:.1f}%")
    print(f"  Cost: ~3 LLM calls per invoice (3x baseline)")
    print(f"  Tradeoff: 3x cost for consensus filtering")
    return majority_extractor, score

def run_gepa(trainset, testset, mode="light"):
    """Stage 4: GEPA instruction optimization."""
    print("\n" + "=" * 60)
    print(f"STAGE 4: + GEPA Instruction Learning (mode={mode})")
    print("=" * 60)

    from dspy.teleprompt import GEPA

    reflection_lm = dspy.LM("openai/gpt-4o-mini", temperature=1.0)

    optimizer = GEPA(
        metric=field_accuracy,
        auto=mode,
        reflection_lm=reflection_lm,
        num_threads=1,
    )

    extractor = optimizer.compile(
        student=dspy.ChainOfThought(InvoiceExtraction),
        trainset=trainset,
    )

    evaluator = Evaluate(devset=testset, metric=field_accuracy, display_progress=False)
    score = float(evaluator(extractor))

    print(f"  Accuracy: {score:.1f}%")
    print(f"  Cost: ~1 LLM call per invoice (GEPA cost is upfront during optimization)")
    print(f"  Tradeoff: higher optimization cost, but inference stays at 1 call")
    return extractor, score

## Demo extraction

In [ ]:
def demo_extraction(extractor, testset):
    """Show extraction results on test examples."""
    print("\n" + "=" * 60)
    print("EXTRACTION DEMO")
    print("=" * 60)

    for i, example in enumerate(testset):
        result = extractor(text=example.text)
        score = field_accuracy(example, result)

        print(f"\nInvoice {i + 1} (score: {score:.2f}):")
        for field in FIELD_KEYS:
            gold = getattr(example, field, "") or ""
            got = getattr(result, field, "") or ""
            match = "ok" if gold.strip().lower() == got.strip().lower() else "MISS"
            if gold or got:
                print(f"  {field:20s}: {got:30s} [{match}]")

## Run the demo

The code below mirrors the `main()` block in the source script with the argparse CLI branches removed — it runs the full demo path end-to-end.

In [ ]:
trainset, testset = build_dataset()

# Progressive pipeline
results = {}

_, results["baseline"] = run_baseline(trainset, testset)

_, results["fewshot"] = run_fewshot(trainset, testset)
_, results["majority"] = run_majority(trainset, testset)
extractor, results["gepa"] = run_gepa(trainset, testset, mode="light")

# Summary
print("\n" + "=" * 60)
print("PROGRESSIVE IMPROVEMENT SUMMARY")
print("=" * 60)
print(f"  Baseline:              {results['baseline']:5.1f}%  (1 call/invoice)")
print(f"  + Few-Shot:            {results['fewshot']:5.1f}%  (1 call/invoice)")
print(f"  + Majority (3 calls):  {results['majority']:5.1f}%  (3 calls/invoice)")
print(f"  + GEPA:                {results['gepa']:5.1f}%  (1 call/invoice)")
print()
print("  GEPA achieves the best accuracy/cost ratio:")
print("  same inference cost as baseline, better accuracy than majority voting.")

# Show extractions with the best model
demo_extraction(extractor, testset)